# colony4_report

> Phase 18, Stage 4 (`specs/multiagent/phase18/stage4.md`, FR-A24): the analysis of guards that scale. `stage4Summaries`: H18-13 to H18-15 (each option minus assigned guards at 64 agents, honest pooled share, one-sided exact sign-flip, Holm over the three) with their robustness on raw honest welfare; the per-seed difference-in-differences from 16 to 64 agents; each arm's advantage over the plain board; squads at 64 against guards at 8; the clean cost at 16; the combined arms; per-cell means of the mechanism measures. `writeStage4` writes the report.

In [ ]:
#| default_exp colony4_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Summaries

In [ ]:
#| export
import numpy as np

from explRL.bench_stats import holm
from explRL.colony4 import STAGE4_PLAN
from explRL.colony_report import NAN, bootSeeds, cellVals, signFlipExact

HYP4 = (('H18_13', 'guardDeal'), ('H18_14', 'failFlag'), ('H18_15', 'squads'))
CELL_KEYS = ('honestPooled', 'welfareHonest', 'contamination', 'guardHeldLies', 'contestRateGuarded', 'contestRateUnguarded',
             'contestLagGuarded', 'usesPerLieGuarded', 'usesPerLieUnguarded', 'usesPerUser', 'openersPerLie', 'flagsPerLie',
             'flagsOnTrue', 'usesAfterFlag', 'guardPostsPerTurn', 'guardUnposted', 'inTokPerCall')


def diffs(rows, arm, base, metric, fault, N, Nb=None):
    "{(seed, N): arm at N - base at Nb (default N)}, paired on seed."
    a, b = cellVals(rows, arm, fault, metric, [N]), cellVals(rows, base, fault, metric, [Nb or N])
    return {(s, N): a[(s, N)] - b[(s, Nb or N)] for s, _ in sorted(a) if (s, Nb or N) in b
            and np.isfinite(a[(s, N)]) and np.isfinite(b[(s, Nb or N)])}


def _two(v):
    return min(1.0, 2 * min(signFlipExact(v, 'greater'), signFlipExact(v, 'less'))) if len(v) else NAN


def _test(dp, nboot, tail='greater'):
    est, lo, hi = bootSeeds(dp, nboot=nboot)
    v = list(dp.values())
    return dict(n=len(dp), estimate=est, ci=[lo, hi], p=signFlipExact(v) if tail == 'greater' else _two(v))


def stage4Summaries(rows4, rowsReused, plan=STAGE4_PLAN, nboot=2000):
    "FR-A24: primaries with Holm and robustness, difference-in-differences, advantages, squads against 8, clean cost, cells."
    rows, ind = list(rows4) + list(rowsReused), 'independent'
    hyp = {}
    for h, arm in HYP4:
        t = _test(diffs(rows, arm, 'colonyGuards', 'honestPooled', ind, 64), nboot)
        rob = _test(diffs(rows, arm, 'colonyGuards', 'welfareHonest', ind, 64), nboot)
        hyp[h] = dict(t, arm=arm, robustness=rob, robust=bool(np.sign(rob['estimate']) == np.sign(t['estimate'])))
    for h, q in zip([h for h, _ in HYP4], holm([hyp[h]['p'] for h, _ in HYP4])):
        hyp[h].update(pHolm=float(q), supported=bool(q < plan['primary']['alpha'] and hyp[h]['estimate'] > 0))
    did = {}
    for _, arm in HYP4:
        d64, d16 = (diffs(rows, arm, 'colonyGuards', 'honestPooled', ind, N) for N in (64, 16))
        did[arm] = _test({(s, 0): v - d16[(s, 16)] for (s, _), v in d64.items() if (s, 16) in d16}, nboot, 'two')
    arms = ['colonyGuards'] + [a for a in plan['arms']]
    adv = {a: {N: float(np.mean(list(dp.values()))) for N in (16, 64)
               if (dp := diffs(rows, a, 'plain', 'honestPooled', ind, N))} for a in arms}
    sq8 = _test(diffs(rows, 'squads', 'colonyGuards', 'honestPooled', ind, 64, 8), nboot, 'two')
    clean = {a: _test(diffs(rows, a, 'colonyGuards', 'honestPooled', 'clean', 16), nboot, 'two') for a in ('guardDeal', 'failFlag')}
    combined = {a: _test(diffs(rows, a, 'colonyGuards', 'honestPooled', ind, N), nboot, 'two')
                for a, N in (('guardDealFlag', 16), ('squadsAll', 64))}
    cells = {}
    for r in rows:
        cells.setdefault((r['level'], int(float(r['N']))), []).append(r)
    mean = lambda rs, k: float(np.nanmean([float(r[k]) for r in rs if k in r and r[k] not in ('', None)] or [NAN]))
    cells = {f'{lv} N {N}': {k: mean(rs, k) for k in CELL_KEYS} for (lv, N), rs in sorted(cells.items(), key=lambda x: (x[0][1], x[0][0]))}
    return dict(hyp=hyp, did=did, advantage=adv, squads8=sq8, cleanCost=clean, combined=combined, cells=cells)

## Report

In [ ]:
#| export
_f = lambda x: 'n/a' if x is None or x != x else f'{x:.3f}'
_ci = lambda t: f"[{_f(t['ci'][0])}, {_f(t['ci'][1])}]"


def writeStage4(path, S, meta):
    "report.md for Stage 4."
    L = ['# Phase 18, Stage 4: guards that scale (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| test | n | arm - colonyGuards (N 64) | 95% interval | p | Holm | supported | on raw welfare | robust |',
         '|---|---|---|---|---|---|---|---|---|']
    for h, t in S['hyp'].items():
        L.append(f"| {h} {t['arm']} | {t['n']} | {_f(t['estimate'])} | {_ci(t)} | {_f(t['p'])} | {_f(t['pHolm'])} | "
                 f"{t['supported']} | {_f(t['robustness']['estimate'])} | {t['robust']} |")
    L += ['', '| 16 to 64 agents: (arm - colonyGuards)@64 - @16 | n | estimate | 95% interval | p (two-sided) |', '|---|---|---|---|---|']
    L += [f"| {a} | {t['n']} | {_f(t['estimate'])} | {_ci(t)} | {_f(t['p'])} |" for a, t in S['did'].items()]
    L += ['', 'Advantage over the plain board: ' + '; '.join(
        f"{a} " + ', '.join(f'N {N} {_f(v)}' for N, v in d.items()) for a, d in S['advantage'].items() if d)]
    t = S['squads8']
    L += [f"Squads at 64 minus colonyGuards at 8: {_f(t['estimate'])} {_ci(t)}, p {_f(t['p'])}"]
    L += ['Clean cost at 16 (arm - colonyGuards): ' + '; '.join(f"{a} {_f(t['estimate'])} {_ci(t)}" for a, t in S['cleanCost'].items())]
    L += ['Combined arms against colonyGuards: ' + '; '.join(f"{a} {_f(t['estimate'])} {_ci(t)}" for a, t in S['combined'].items()), '']
    keys = list(next(iter(S['cells'].values())))
    L += ['| cell | ' + ' | '.join(keys) + ' |', '|---|' + '---|' * len(keys)]
    L += [f'| {c} | ' + ' | '.join(_f(v) for v in d.values()) + ' |' for c, d in S['cells'].items()]
    open(path, 'w').write('\n'.join(L) + '\n')

## Tests

In [ ]:
import tempfile
from pathlib import Path

# synthetic rows with known effects: guardDeal +0.10 and failFlag +0.02 (noisy) over colonyGuards at 64; squads -0.05;
# colonyGuards' advantage over plain 0.20 at 16, 0.16 at 64; guardDeal's gain at 16 is 0.04, so its 16-to-64 change is +0.06
rng = np.random.default_rng(0)
def row(arm, fault, N, s, hp):
    return dict(level=f'{arm}|{fault}', N=N, seed=s, honestPooled=hp, welfareHonest=hp * 1.3, contamination=0.5)
R4, RR = [], []
for s in range(1800, 1808):
    e = rng.normal(0, 0.005)
    RR += [row('plain', 'independent', 16, s, 0.02 + e), row('colonyGuards', 'independent', 16, s, 0.22 + e),
           row('plain', 'independent', 64, s, -0.09 + e), row('colonyGuards', 'independent', 64, s, 0.07 + e),
           row('colonyGuards', 'independent', 8, s, 0.28 + e), row('colonyGuards', 'clean', 16, s, 0.36 + e)]
    R4 += [row('guardDeal', 'independent', 64, s, 0.17 + e), row('failFlag', 'independent', 64, s, 0.07 + e + (0.03 if s % 2 else -0.01)),
           row('squads', 'independent', 64, s, 0.02 + e), row('guardDeal', 'independent', 16, s, 0.26 + e),
           row('failFlag', 'independent', 16, s, 0.22 + e), row('squads', 'independent', 16, s, 0.22 + e),
           row('guardDeal', 'clean', 16, s, 0.33 + e), row('failFlag', 'clean', 16, s, 0.36 + e),
           row('guardDealFlag', 'independent', 16, s, 0.27 + e), row('squadsAll', 'independent', 64, s, 0.2 + e)]
S = stage4Summaries(R4, RR, nboot=200)
h = S['hyp']
assert abs(h['H18_13']['estimate'] - 0.10) < 1e-9 and h['H18_13']['p'] == 1 / 256 and h['H18_13']['supported'] and h['H18_13']['robust']
assert abs(h['H18_13']['pHolm'] - 3 / 256) < 1e-12                      # the smallest p of three, times 3
assert abs(h['H18_14']['estimate'] - 0.01) < 1e-9 and h['H18_14']['p'] > 1 / 256
assert h['H18_15']['estimate'] < 0 and not h['H18_15']['supported'] and h['H18_15']['robust']
assert abs(S['did']['guardDeal']['estimate'] - 0.06) < 1e-9 and abs(S['did']['squads']['estimate'] + 0.05) < 1e-9
assert abs(S['advantage']['colonyGuards'][16] - 0.20) < 1e-9 and abs(S['advantage']['colonyGuards'][64] - 0.16) < 1e-9
assert abs(S['squads8']['estimate'] + 0.26) < 1e-9 and abs(S['cleanCost']['guardDeal']['estimate'] + 0.03) < 1e-9
assert abs(S['combined']['squadsAll']['estimate'] - 0.13) < 1e-9 and 'guardDeal|independent N 64' in S['cells']
with tempfile.TemporaryDirectory() as d:
    writeStage4(Path(d) / 'report.md', S, dict(planHash='x', commit='y', mode='quick'))
    txt = (Path(d) / 'report.md').read_text()
assert 'H18_13 guardDeal' in txt and '16 to 64 agents' in txt